# Appendix C.5 — Designations (SSSI / SAC / SPA)

Evidence for the claims made about the Natural England designation layers in Appendix C.

**Sources used**

| file | what it is |
| --- | --- |
| `raw_datasets/Special_Protection_Areas_England.geojson` | SPA source layer |
| `raw_datasets/Special_Areas_of_Conservation_England.geojson` | SAC source layer |
| `raw_datasets/Sites_of_Special_Scientific_Interest_England.geojson` | SSSI source layer |
| `ttl/regulation.ttl` | the water-side points, for the CRS comparison |


In [1]:
import os, json, warnings
from pathlib import Path
import pandas as pd

warnings.filterwarnings("ignore")
ROOT = Path.cwd()
while not (ROOT / "raw_datasets").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
RAW = ROOT / "raw_datasets"
REG = RAW / "access_database_csv_files"

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 90)
print("repository root:", ROOT)


repository root: /Users/waf/git/projects/demonstrator-poc


In [2]:
LAYERS = {
    "SPA":  "Special_Protection_Areas_England.geojson",
    "SAC":  "Special_Areas_of_Conservation_England.geojson",
    "SSSI": "Sites_of_Special_Scientific_Interest_England.geojson",
}
docs = {k: json.loads((RAW / v).read_text()) for k, v in LAYERS.items()}
for k, d in docs.items():
    print(f"{k}: {len(d['features']):,} features, top-level keys {sorted(d.keys())}")


SPA: 195 features, top-level keys ['bbox', 'crs', 'features', 'numberMatched', 'numberReturned', 'timeStamp', 'totalFeatures', 'type']
SAC: 245 features, top-level keys ['bbox', 'crs', 'features', 'numberMatched', 'numberReturned', 'timeStamp', 'totalFeatures', 'type']
SSSI: 146 features, top-level keys ['bbox', 'crs', 'features', 'numberMatched', 'numberReturned', 'timeStamp', 'totalFeatures', 'type']


---
## C.5.1 Two coordinate systems on the same feature, and a CRS declaration that contradicts the geometry

> *"Each layer declares a CRS its own coordinates contradict, and carries a second coordinate system in
> its attributes, with nothing marking either as authoritative."*

Each file carries a `crs` member — deprecated by RFC 7946, which specifies that GeoJSON coordinates are
CRS84 longitude/latitude — and it names something else:


In [3]:
for k, d in docs.items():
    print(f"{k:5} crs member: {json.dumps(d.get('crs'))}")
    geom = d["features"][0]["geometry"]
    coord = geom["coordinates"]
    while isinstance(coord[0], list):
        coord = coord[0]
    print(f"      first coordinate: {coord}")


SPA   crs member: {"type": "name", "properties": {"name": "urn:ogc:def:crs:EPSG::4326"}}
      first coordinate: [-1.167102, 50.852577]
SAC   crs member: {"type": "name", "properties": {"name": "urn:ogc:def:crs:EPSG::4326"}}
      first coordinate: [-2.476926, 50.885684]
SSSI  crs member: {"type": "name", "properties": {"name": "urn:ogc:def:crs:EPSG::4326"}}
      first coordinate: [-2.53607, 50.835636]


`EPSG::4326` is latitude-then-longitude by its own definition; the coordinates are
longitude-then-latitude, i.e. CRS84. The declaration and the data disagree about axis order. A consumer
that honours the declared CRS plots England in the Indian Ocean; one that ignores it and follows RFC 7946
is right — but only by ignoring what the file says.

Now the second coordinate system, in the *attributes* of the very same feature:


In [4]:
props = docs["SPA"]["features"][0]["properties"]
geom_coord = docs["SPA"]["features"][0]["geometry"]["coordinates"]
while isinstance(geom_coord[0], list):
    geom_coord = geom_coord[0]
print("SPA feature, geometry  :", geom_coord, " (degrees, lon/lat)")
print("SPA feature, attributes:", {k: props[k] for k in ("easting","northing","latitude","longitude","grid_ref")
                                   if k in props})
print("\nThe same site carries British National Grid metres AND WGS84 degrees, in different places,")
print("with nothing saying which is authoritative.")


SPA feature, geometry  : [-1.167102, 50.852577]  (degrees, lon/lat)
SPA feature, attributes: {'easting': 470218.8808, 'northing': 93525.1488, 'latitude': '50:44:13N', 'longitude': '1:00:22W', 'grid_ref': 'SZ702935'}

The same site carries British National Grid metres AND WGS84 degrees, in different places,
with nothing saying which is authoritative.


### What this costs a consumer

Mixing coordinate reference systems is not itself a defect: the EA publishes water data in EPSG:27700,
Natural England publishes these layers in degrees, and any full GeoSPARQL engine (GraphDB, Stardog,
Virtuoso) reprojects between them as a matter of course. The bundled pyoxigraph store does not — that is
a limit of this demonstrator's engine, not of the data.

What *is* a defect is the declaration, because reprojection is only as good as the CRS it is told. An
engine that honours `EPSG::4326` as written applies latitude-then-longitude to coordinates that are
longitude-then-latitude; the numbers survive, the axes swap, and every feature lands somewhere
plausible and wrong. Below, the two coordinate systems this project actually has to reconcile:


In [5]:
import pyoxigraph as ox

store = ox.Store()
store.bulk_load(path=str(ROOT / "ttl" / "regulation.ttl"), format=ox.RdfFormat.TURTLE)
pt = [r[0].value for r in store.query("""
PREFIX geo: <http://www.opengis.net/ont/geosparql#>
PREFIX reg: <http://environment.data.gov.uk/ontology/regulation/>
SELECT ?w WHERE { ?d a reg:DischargePoint ; geo:hasGeometry ?g . ?g geo:asWKT ?w
                  FILTER(CONTAINS(STR(?w), "27700")) } LIMIT 1""")][0]
print("a discharge point, as the EA publishes it:")
print("  ", pt)
print()
import re
e, n = map(float, re.search(r"POINT\(([\d.]+) ([\d.]+)\)", pt).groups())
print(f"as British National Grid metres, which is what it is: easting {e:,.0f} m, northing {n:,.0f} m")
print()
print("The designation layers, for the same ground:", docs["SPA"].get("bbox"))
print()
print("Reconciling the two is a reprojection, and a full GeoSPARQL engine does it unasked.")
print("What no engine can repair is a CRS declared with the wrong axis order: honour EPSG::4326")
print("as written and a site at lon -2.0, lat 50.7 is read as lon 50.7, lat -2.0 -- a point in the")
print("South Atlantic, produced without error, from coordinates that were correct on the way in.")


a discharge point, as the EA publishes it:
   <http://www.opengis.net/def/crs/EPSG/0/27700> POINT(400710 93560)

as British National Grid metres, which is what it is: easting 400,710 m, northing 93,560 m

The designation layers, for the same ground: [-2.621178, 50.561099, -0.603797, 50.97242]

Reconciling the two is a reprojection, and a full GeoSPARQL engine does it unasked.
What no engine can repair is a CRS declared with the wrong axis order: honour EPSG::4326
as written and a site at lon -2.0, lat 50.7 is read as lon 50.7, lat -2.0 -- a point in the
South Atlantic, produced without error, from coordinates that were correct on the way in.


In [6]:
# The bounding boxes, side by side -- the numbers do not even overlap.
spa_bbox = docs["SPA"].get("bbox")
print("designation layer bbox (degrees):", spa_bbox)
print("water-side coordinates (metres) : easting ~%s, northing ~%s" % (int(e), int(n)))


designation layer bbox (degrees): [-2.621178, 50.561099, -0.603797, 50.97242]
water-side coordinates (metres) : easting ~400710, northing ~93560


---
## C.5.2 Sites are published as multipart geometries with no single feature per site

> *"a consumer must dissolve by name to get one site per site."*


In [7]:
rows = []
for k, d in docs.items():
    name_key = {"SPA": "spa_name", "SAC": "sac_name", "SSSI": "sssi_name"}.get(k)
    if name_key is None or name_key not in d["features"][0]["properties"]:
        name_key = next(p for p in d["features"][0]["properties"] if "name" in p.lower())
    names = pd.Series([f["properties"].get(name_key) for f in d["features"]])
    per_site = names.value_counts()
    rows.append({"layer": k, "name field": name_key, "features": len(names),
                 "distinct sites": names.nunique(),
                 "sites split across >1 feature": int((per_site > 1).sum()),
                 "worst case": f"{per_site.iloc[0]} features ({per_site.index[0]})"})
pd.DataFrame(rows)


,layer,name field,features,distinct sites,sites split across >1 feature,worst case
0,SPA,spa_name,195,6,4,177 features (Dorset Heathlands)
1,SAC,sac_name,245,15,11,145 features (Dorset Heaths)
2,SSSI,name,146,146,0,1 features (Batcombe Down SSSI)


In [8]:
names = pd.Series([f["properties"].get("spa_name") for f in docs["SPA"]["features"]])
print("Dorset Heathlands, as published -- 177 separate features for one designated site:")
print(names.value_counts().head(6).to_string())
print("\nCounting features counts polygons, not sites. Any 'how many SPAs are here' question")
print("answered from the row count is wrong by a factor of ~30 for this layer.")


Dorset Heathlands, as published -- 177 separate features for one designated site:
Dorset Heathlands           177
Avon Valley                   8
New Forest                    4
Poole Harbour                 4
Solent and Dorset Coast       1
Chesil Beach & the Fleet      1

Counting features counts polygons, not sites. Any 'how many SPAs are here' question
answered from the row count is wrong by a factor of ~30 for this layer.


In [9]:
# The codes, by contrast, are clean -- which is what makes the dissolve safe.
spa = pd.DataFrame([f["properties"] for f in docs["SPA"]["features"]])
print("distinct spa_name values :", spa.spa_name.nunique())
print("distinct spa_code values :", spa.spa_code.nunique())
print("name-to-code mapping is 1:1:", (spa.groupby("spa_name").spa_code.nunique() == 1).all())
spa[["spa_name","spa_code"]].drop_duplicates().sort_values("spa_name")


distinct spa_name values : 6
distinct spa_code values : 6
name-to-code mapping is 1:1: True


,spa_name,spa_code
2,Avon Valley,UK9011091
1,Chesil Beach & the Fleet,UK9010091
14,Dorset Heathlands,UK9010101
9,New Forest,UK9011031
22,Poole Harbour,UK9010111
0,Solent and Dorset Coast,UK9020330
